# ECS7001P Assignment 2: Multimodal Question Answering

**Project Overview: Building a Custom LLaVA Architecture**

In this assignment, you will tackle a **Multimodal Question Answering (MQA)** task using a proprietary "in-house" corpus. This dataset consists of images paired with multiple-choice questions (MCQs). Your objective is to architect and train a customized **LLaVA (Large Language-and-Vision Assistant)** model by integrating a specialized vision encoder with a lightweight Large Language Model (LLM).



In [1]:
!pip install transformers

Defaulting to user installation because normal site-packages is not writeable


**Dataset Description**

The dataset consists of **11,600 unique images**, each paired with 5–6 MCQs, totaling approximately **65k questions**.

**Data Splits**

* **Training Set:** 10k images and 56k MCQs. (Alignment is already provided).
* **Validation Set:** 600 images and 3.3k MCQs.
* **Test Set:** 1k images and 5.5k MCQs.

**File Structure**
The data is organized into an image repository and two types of `.jsonl` metadata files:

* **images/train/**: Training image set.
* **images/test_val/**: Combined images for both validation and testing.
* **vqa_xxx_student.jsonl**: Contains the MCQs, multiple-choice options, and metadata.
* **caption_xxx_student.jsonl**: Contains the image captions used to map images back to their respective questions.

**Note:** There is no caption file for the **Training** set, as the image-to-MCQ alignment is already provided for your supervised learning phase. Further **vqa_val_teacher.jsonl** were provided for you to evaluate the system's performace in the development phrase.

In [2]:
!curl -L 'https://huggingface.co/datasets/juntaoyu/ECS7001P_Assignment2_Data/resolve/main/ECS7001P_Assignment2_data.zip' -o ECS7001P_Assignment2_data.zip
!unzip -o ECS7001P_Assignment2_data.zip -x '__MACOSX/*'

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1411  100  1411    0     0  10347      0 --:--:-- --:--:-- --:--:-- 10375
100  108M  100  108M    0     0   232M      0 --:--:-- --:--:-- --:--:--  232M
Archive:  ECS7001P_Assignment2_data.zip
  inflating: images/.DS_Store        
  inflating: images/train/000835105.jpg  
  inflating: images/train/000699806.jpg  
  inflating: images/train/000020362.jpg  
  inflating: images/train/000841684.jpg  
  inflating: images/train/001560854.jpg  
  inflating: images/train/001171576.jpg  
  inflating: images/train/001467691.jpg  
  inflating: images/train/000668633.jpg  
  inflating: images/train/001161771.jpg  
  inflating: images/train/000377296.jpg  
  inflating: images/train/000193816.jpg  
  inflating: images/train/000754860.jpg  
  inflating: images/train/000419066.jpg  
  inflating: images/train/001517410.jpg  
  inflating: image

In [3]:
import torch, os, json
from transformers import CLIPModel, CLIPProcessor,CLIPVisionConfig
from transformers.image_utils import load_image

[HAMI-core Msg(3785:140488778349888:libvgpu.c:839)]: Initializing.....
/opt/conda/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**Technical Specifications**
You will be working with the following components:

*    **Vision Encoder:** AlpachinoNLP/LongCLIP-ViT-B-32 (Optimized for long-form captioning and extended visual context).

*    **Language Backbone:** Qwen3-0.6b (A high-efficiency, small-scale LLM).

*    **The Goal:** Align visual features with linguistic embeddings to enable the model to "reason" over images to answer complex MCQs.
**Note:** You are **NOT** allowed to use any other pretrained models to complete this assingment.

In [4]:
CLIP_MODEL_ID = "AlpachinoNLP/LongCLIP-ViT-B-32"
TEXI_MODEL_ID = "Qwen/Qwen3-0.6B"

**Task 1: Image-MCQ Reconstruction**

Before training the LLaVA model, you must reconstruct the image-MCQ mappings for the validation and test sets. Since the `test_val` folder provides images in bulk, you will use the **LongCLIP** vision head and the provided captions to perform semantic alignment.

**Objectives:**
* Implement the `recover_image_id()` method to perform semantic alignment between image features and captions.
* Utilize the `caption_ids` key present in the relevant `.jsonl` files to link captions and MCQs.
* Recover the `image` field in `vqa_val_student.jsonl` and `vqa_test_student.jsonl`. This field must store the relative path (e.g., `"images/test_val/000957012.jpg"`).
* Save the resulting files as `vqa_val_student_with_image_ids.jsonl` and `vqa_test_student_with_image_ids.jsonl`.

**Evaluation:**

The provided **LongCLIP** model is pre-trained on an extended version of this corpus and is highly accurate. A successful implementation should achieve **>99% accuracy** when evaluated using the provided `evaluate_image_caption_alignment()` method.

*Note: Feel free to add helper methods, ensuring they are clearly commented for clarity.*

In [5]:
#function given to evaluate the image-caption alignment task
def evaluate_image_caption_alignment(gold_path,pred_path):
  gold_data = set()
  pred_data = set()
  for line in open(gold_path, 'r'):
    jdata = json.loads(line)
    gold_data.add((jdata['caption_id'],jdata['image']))
  for line in open(pred_path, 'r'):
    jdata = json.loads(line)
    pred_data.add((jdata['caption_id'],jdata['image']))
  return len(gold_data & pred_data)/len(gold_data)

In [6]:
#imports:
from pathlib import Path
from PIL import Image
import torch.nn.functional as F
from scipy.optimize import linear_sum_assignment
from tqdm.auto import tqdm

###HELPER FUNCTIONS:
#- read_jsonl: function to read jsonl files
#- write_jsonl: function to write jsonl files
#- encode_texts: function to encode texts using the CLIP text encoder
#- encode_images: function to encode images using the CLIP image encoder

#function to read and write jsonl files
def read_jsonl(path):
    data = []
    with open(path, 'r') as f:
        for line in f:
            data.append(json.loads(line))
    return data

def write_jsonl(path, data):
    with open(path, 'w') as f:
        for item in data:
            f.write(json.dumps(item) + '\n')
            
#function to encode texts
def encode_texts(model, processor, texts, device, batch_size=32):
    features = []
    model.eval()
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="Encoding captions"):
            batch_texts = texts[i:i+batch_size]
            inputs = processor(text=batch_texts, return_tensors="pt", padding=True, truncation=True)
            inputs = {k: v.to(device) for k, v in inputs.items()}

            #forward through text encoder
            text_outputs = model.text_model(**inputs)

            #CLS pooled repre
            pooled_output = text_outputs.pooler_output

            #projection into CLIP embedding space
            text_features = model.text_projection(pooled_output)
            #normalize
            text_features = F.normalize(text_features, p=2, dim=1)
            features.append(text_features.cpu())
    return torch.cat(features, dim=0)
    
#function to encode images
def encode_images(model, processor, image_paths, device, batch_size=16):
    features = []
    model.eval()
    with torch.no_grad():
        for i in tqdm(range(0, len(image_paths), batch_size), desc="Encoding images"):
            batch_paths = image_paths[i:i+batch_size]
            images = [Image.open(path).convert("RGB") for path in batch_paths]
            inputs = processor(images=images, return_tensors="pt")
            pixel_values = inputs['pixel_values'].to(device)
            vision_outputs = model.vision_model(pixel_values)
            pooled_output = vision_outputs.pooler_output
            image_features = model.visual_projection(pooled_output)
            #need to normalize the features to compute cosine similarity later
            image_features = F.normalize(image_features, p=2, dim=1)
            features.append(image_features.cpu())
    return torch.cat(features, dim=0)

In [7]:
#turns unlabeled captions and unlabeled images into a correct 1:1 mapping, then writes those image IDs into the VQA data.
def recover_image_id():
  device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
  #load the CLIP model and processor
  model = CLIPModel.from_pretrained(CLIP_MODEL_ID).to(device)
  processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID)
  
  #read json validation and test captions
  #each item has caption and caption_id
  caption_val = read_jsonl("caption_val_student.jsonl")
  caption_test = read_jsonl("caption_test_student.jsonl")
  all_captions = caption_val + caption_test

  #get image paths
  image_dir = Path("images/test_val")
  image_paths = sorted([str(path) for path in image_dir.glob("*.jpg")])
  
  #check if the number of captions matches the number of images
  assert len(all_captions) == len(image_paths), "Number of captions and images do not match!"
  
  #encode captions and images
  text_features = encode_texts(model, processor, [item['caption'] for item in all_captions], device)
  image_features = encode_images(model, processor, image_paths, device)
  
  #compute cosine similarity between text and image features. caption-image similarity
  similarity = text_features @ image_features.T
  #use linear sum assignment to find the best matching between captions and images
  row_ind, col_ind = linear_sum_assignment(similarity.cpu().numpy(), maximize=True)
  
  #creates dictionary mapping caption_id to image path based on the matching results
  caption_to_image = {
      all_captions[r]['caption_id']: image_paths[c]
      for r, c in zip(row_ind, col_ind)
  }

  #loop over VQA val and test files
  for in_path, out_path in [
      ('vqa_val_student.jsonl', 'vqa_val_student_with_image_ids.jsonl'),
      ('vqa_test_student.jsonl', 'vqa_test_student_with_image_ids.jsonl'),
  ]:
    #read it
    rows = read_jsonl(in_path)
    #for each row, fills row[image] using caption_id
    for row in rows:
      row['image'] = caption_to_image[row['caption_id']]
    #write it back to jsonl file
    write_jsonl(out_path, rows)
  
  #final evaluation. compare reconstructed mapping to teacher val file
  acc = evaluate_image_caption_alignment('vqa_val_teacher.jsonl','vqa_val_student_with_image_ids.jsonl')
  print(f"Evaluation accuracy: {acc*100.0:.2f}%")

In [8]:
recover_image_id()

[HAMI-core Msg(3785:140488778349888:libvgpu.c:855)]: Initialized
Encoding images: 100%|██████████| 100/100 [00:18<00:00,  5.30it/s]

Evaluation accuracy: 99.67%


### Task 2: LLaVA Training & Evaluation
Once the mappings are restored, you will implement the LLaVA framework via the following steps:

1. **Model Construction:** Extract the vision head from **LongCLIP** and integrate it with **Qwen** to build your custom LLaVA model within the `train_evaluate_LLaVA_model()` method.
2. **Alignment:** Implement the `LlavaMultiModalProjector` class to map CLIP visual tokens into the Qwen3 embedding space using a projection layer (Linear or MLP).
3. **Fine-tuning:** Train the system to process an image and a text-based MCQ simultaneously to predict the correct answer using the `train_evaluate_LLaVA_model()` method.
4. **Inference:** Evaluate performance on the reconstructed validation and test pipelines using the `get_prediction()` and `evaluation()` methods.
5. **Output:** Export your final predictions into JSONL files named `vqa_val_student_final.jsonl` and `vqa_test_student_final.jsonl`.

In [9]:
#imports given
import random,json,datetime
import torch
import numpy as np
from PIL import Image
from torch import nn
from transformers import (AutoTokenizer,
                          CLIPImageProcessor,
                          AutoModelForCausalLM,
                          CLIPVisionModel,
                          LlavaForConditionalGeneration,
                          LlavaProcessor,
                          CLIPVisionConfig,
                          LlavaConfig)
from transformers.activations import ACT2FN
from tqdm.auto import tqdm

In [10]:
#function given:
#evaluate the vqa accuracy between the gold and predicted data
def evaluation(gold_path,pred_path):
  gold_data = []
  pred_data = []
  for line in open(gold_path, 'r'):
    jdata = json.loads(line)
    gold_data.append(jdata['answer'])
  for line in open(pred_path, 'r'):
    jdata = json.loads(line)
    pred_data.append(jdata['answer'])
  return sum(1 for i in range(len(gold_data)) if gold_data[i] == pred_data[i])/len(gold_data)


#get the predictions from the model for evaluation
def get_prediction(model,processor,batch_size,device,dataset):
  #Your code for Task 2.4
  tokenizer = processor['tokenizer']
  image_processor = processor['image_processor']
  id_to_answer = processor['id_to_answer']
  choice_token_ids = processor['choice_token_ids']

  model.eval()
  predictions = []
  
  #loop over the dataset in batches, process the images and prompts, and get the models predictions
  for start in tqdm(range(0, len(dataset['image_paths']), batch_size), desc='Predicting'):
    end = min(start + batch_size, len(dataset['image_paths']))
    batch_images = [Image.open(p).convert('RGB') for p in dataset['image_paths'][start:end]]
    batch_prompts = dataset['prompts'][start:end]

    #tokenize the prompts and process the images for the model
    text_inputs = tokenizer(
        batch_prompts,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=min(512, tokenizer.model_max_length),
    )
    image_inputs = image_processor(images=batch_images, return_tensors='pt')

    input_ids = text_inputs['input_ids'].to(device)
    attention_mask = text_inputs['attention_mask'].to(device)
    pixel_values = image_inputs['pixel_values'].to(device)

    #get the models output logits and extract the scores for the answer choices
    with torch.no_grad():
      outputs = model(pixel_values=pixel_values, input_ids=input_ids, attention_mask=attention_mask)
      next_token_logits = outputs.logits[:, -1, :]

      choice_scores = torch.stack([next_token_logits[:, tid] for tid in choice_token_ids], dim=1)
      pred_ids = choice_scores.argmax(dim=1).tolist()
      predictions.extend(pred_ids)
    
  return predictions

#output the predictions to the files
def write_answers_to_file(path, predictions,answer_to_ids):
  #Your code for Task 2.5
  id_to_answer = {v: k for k, v in answer_to_ids.items()}
  output_path = path.replace("_with_image_ids", "_final")
  
  #read the original file, replace the answer with the predicted answer, and write to a new file: "_final"
  rows=[]
  for i, line in enumerate(open(path, 'r')):
    jdata = json.loads(line)
    pred = predictions[i]
    if isinstance(pred, int):
      jdata['answer'] = id_to_answer[pred]
    else:
      jdata['answer'] = pred
    rows.append(jdata)
    
  with open(output_path, 'w') as f:
    for row in rows:
      f.write(json.dumps(row) + '\n')
      
  print(f"Output written to {output_path}")


In [11]:
#The projection layer for LLaVA
class LlavaMultiModalProjector(nn.Module):
  #Your code for Task 2.2
  def __init__(self, config: LlavaConfig):
    super().__init__()
    vision_hidden = config.vision_config.hidden_size
    text_hidden = config.text_config.hidden_size
    act_name = getattr(config, 'projector_hidden_act', 'gelu')

    #MLP linear + activation + linear
    self.linear_1 = nn.Linear(vision_hidden, text_hidden)
    act= ACT2FN[act_name] if act_name in ACT2FN else ACT2FN["gelu"]
    if isinstance(act, type): #if it's a class, instantiate it
      act = act()
    self.act = act
    self.linear_2 = nn.Linear(text_hidden, text_hidden)
  
  #passes image features through MLP
  def forward(self, image_features):
    x = self.linear_1(image_features)
    x = self.act(x)
    x = self.linear_2(x)
    return x


In [12]:
#load the image paths and create prompt from MCQ
#function given
def load_image_vqa_data(path,answer_to_ids):
  image_paths = []
  prompts = []
  answers = []
  for line in open(path, 'r'):
    jdata = json.loads(line)
    image_paths.append(jdata['image'])
    question = jdata['question']
    options = jdata['options']
    answer = answer_to_ids.get(jdata.get('answer', ""), None)

    prompt = (
        "<image>\nAnswer with A, B, C, or D only.\n"
        f"Question: {question}\n"
        "Options:\n"
        f"A. {options[0]}\n"
        f"B. {options[1]}\n"
        f"C. {options[2]}\n"
        f"D. {options[3]}\n"
        "Answer:\n"
    )

    prompts.append(prompt)
    answers.append(answer)
  return {'image_paths':image_paths, 'prompts':prompts, 'answers':answers}

##custom LLaVA model
class CustomLLaVAForMCQ(nn.Module):
  def __init__(self, vision_encoder, language_model, projector):
    super().__init__()
    self.vision_encoder = vision_encoder
    self.language_model = language_model
    self.projector = projector
  
  #encode_image: takes in raw pixel values -> vision encoder -> projection layer -> projected features
  def encode_image(self, pixel_values):
    image_features = self.vision_encoder(pixel_values=pixel_values)
    #if it has already a pooler output, use it. Otherwise, use the CLS token representation as the pooled output
    if hasattr(image_features, "pooler_output") and image_features.pooler_output is not None:
      pooled = image_features.pooler_output
    else:
      pooled = image_features.last_hidden_state[:, 0, :]
    
    projected_features = self.projector(pooled)
    return projected_features.unsqueeze(1)
  
  #image + prompt -> encode image and text -> concatenate -> language model -> output logits
  def forward(self, pixel_values, input_ids, attention_mask, labels=None):
    lm_dtype = self.language_model.get_input_embeddings().weight.dtype #force image/text embeddings same dtype, was having errors
    
    image_embeds = self.encode_image(pixel_values).to(dtype=lm_dtype)
    text_embeds = self.language_model.get_input_embeddings()(input_ids).to(dtype=lm_dtype)
    #concatenate image and text embeddings along the sequence dimension
    combined_embeds = torch.cat([image_embeds, text_embeds], dim=1)
    
    image_attention= torch.ones((attention_mask.size(0), 1), dtype=attention_mask.dtype, device=attention_mask.device)
    full_attention = torch.cat([image_attention, attention_mask], dim=1)
    
    full_labels = None
    if labels is not None:
      image_labels = torch.full((labels.size(0), 1), -100, dtype=labels.dtype, device=labels.device)
      full_labels = torch.cat([image_labels, labels], dim=1)
    
    outputs = self.language_model(inputs_embeds=combined_embeds, attention_mask=full_attention, labels=full_labels)
    return outputs


#HELPER FUNCTION:
#loads and preprocess images into pixel_values
#tokenizes text prompts into input_ids and attention_mask
#creates labels for the answer choices
#masks the labels for the image tokens with -100 so they are ignored in loss computation
def prepare_training_batch(samples, tokenizer, image_processor, answer_to_ids, device, max_length=256):
  id_to_answer = {v: k for k, v in answer_to_ids.items()}
  texts = []
  prompt_lengths = []
  images = []
  
  for sample in samples:
    answer_letter = id_to_answer[sample['answer']]
    prompt = sample['prompt'] + ' ' + answer_letter
    texts.append(prompt)
    
    prompt_token_ids = tokenizer(sample['prompt'], add_special_tokens=False).input_ids
    prompt_lengths.append(min(len(prompt_token_ids), max_length-1))
    
    images.append(Image.open(sample['image_path']).convert('RGB'))
  
  text_inputs = tokenizer(texts, return_tensors='pt', padding=True, truncation=True, max_length=max_length, add_special_tokens=False)
  image_inputs = image_processor(images=images, return_tensors='pt')
  
  input_ids = text_inputs['input_ids']
  attention_mask = text_inputs['attention_mask']
  labels = input_ids.clone()
  
  for i, prompt_len in enumerate(prompt_lengths):
    labels[i, :prompt_len] = -100
    labels[i, attention_mask[i]==0] = -100
    
  return (
    image_inputs['pixel_values'].to(device),
    input_ids.to(device),
    attention_mask.to(device),
    labels.to(device)
  )


def train_evaluate_LLaVA_model():
  random.seed(0)
  torch.manual_seed(0)
  np.random.seed(0)

  #Your code for Task 2.1 and 2.3
  epochs = 3
  train_batch_size = 2
  eval_batch_size = 2
  max_train_samples = 3000
  learning_rate = 1e-4
  weight_decay = 0.01
  warmup_ratio = 0.1
  grad_accum_steps = 4
  
  answer_to_ids = {"A": 0, "B": 1, "C": 2, "D": 3}
  id_to_answer = {v: k for k, v in answer_to_ids.items()}
  
  if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
  else:
    device = torch.device('cpu')
    print("Using CPU")
    
  tokenizer = AutoTokenizer.from_pretrained(TEXI_MODEL_ID)
  image_processor = CLIPImageProcessor.from_pretrained(CLIP_MODEL_ID)
  vision_encoder = CLIPVisionModel.from_pretrained(CLIP_MODEL_ID)
  language_model = AutoModelForCausalLM.from_pretrained(TEXI_MODEL_ID)
  
  llava_config = LlavaConfig(
    vision_config=vision_encoder.config.to_dict(),
    text_config=language_model.config.to_dict(),
    projector_hidden_act='gelu',
  )
  
  projector = LlavaMultiModalProjector(llava_config)
  model = CustomLLaVAForMCQ(vision_encoder, language_model, projector).to(device)
  
  #freeze heavy backbones, train projector only
  for p in model.vision_encoder.parameters():
    p.requires_grad = False
  for p in model.language_model.parameters():
    p.requires_grad = False
  
  train_data = load_image_vqa_data('vqa_train_student.jsonl', answer_to_ids)
  val_data = load_image_vqa_data('vqa_val_student_with_image_ids.jsonl', answer_to_ids)
  test_data = load_image_vqa_data('vqa_test_student_with_image_ids.jsonl', answer_to_ids)
  
  max_train_samples = min(max_train_samples, len(train_data['image_paths']))
  train_samples = [
    {
      'image_path': train_data['image_paths'][i],
      'prompt': train_data['prompts'][i],
      'answer': train_data['answers'][i]
    }
    for i in range(max_train_samples)
    if train_data['answers'][i] is not None
  ]
  
  trainable_params = [p for p in model.parameters() if p.requires_grad]
  optimizer = torch.optim.AdamW(trainable_params, lr=learning_rate, weight_decay=weight_decay)
  total_steps = max(1, (len(train_samples) // train_batch_size) * epochs // grad_accum_steps)
  warmup_steps = int(total_steps * warmup_ratio)
  
  def lr_lambda(current_step):
    if current_step < warmup_steps:
      return float(current_step) / float(max(1, warmup_steps))
    progress = float(current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    return 0.5 * (1.0 + np.cos(np.pi * progress))
  
  scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
  
  model.train()
  
  for epoch in range(epochs):
    random.shuffle(train_samples)
    running_loss = 0.0
    
    for step, start in enumerate(tqdm(range(0, len(train_samples), train_batch_size), desc=f"Epoch {epoch+1}")):
      batch_samples = train_samples[start:start+train_batch_size]
      pixel_values, input_ids, attention_mask, labels = prepare_training_batch(batch_samples, tokenizer, image_processor, answer_to_ids, device)
      
      outputs = model(pixel_values=pixel_values, input_ids=input_ids, attention_mask=attention_mask, labels=labels)
      loss = outputs.loss / grad_accum_steps
      loss.backward()
      running_loss += loss.item()
      
      if (step + 1) % grad_accum_steps == 0:
        torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        
    print(f"Epoch {epoch+1} completed. Average Loss: {running_loss / max(1, step +1):.4f}")
    
  processor = {
    'tokenizer': tokenizer,
    'image_processor': image_processor,
    'id_to_answer': id_to_answer,
    'choice_token_ids': [tokenizer.encode(' ' + x, add_special_tokens=False)[-1] for x in ['A', 'B', 'C', 'D']],
  }
  
        
  #GIVEN code to get predictions and write to file
  #+ evaluate on val set and print accuracy      
  final_val_predictions = get_prediction(model,processor,eval_batch_size,device,val_data)
  final_test_predictions = get_prediction(model,processor,eval_batch_size,device,test_data)
  
  write_answers_to_file('vqa_val_student_with_image_ids.jsonl',final_val_predictions,answer_to_ids)
  write_answers_to_file('vqa_test_student_with_image_ids.jsonl',final_test_predictions,answer_to_ids)
  
  val_eval_acc = evaluation('vqa_val_teacher.jsonl','vqa_val_student_final.jsonl')
  print(f"Final evaluation accuracy on val set: {val_eval_acc*100.0:.2f}%")
  return val_eval_acc

torch.cuda.empty_cache()
#train_evaluate_LLaVA_model() #UNCOMMENT TO RUN BASELINE


**Task 3: Performance Optimization**

The final goal is to improve the model's reasoning capabilities and MCQ accuracy. You are encouraged to experiment with advanced training and architectural strategies.

**Potential Areas for Improvement:**
* **Architecture Tweaks:** Experiment with more advanced projectors beyond a basic MLP.
* **Fine-tuning Techniques:** Implement **LoRA (Low-Rank Adaptation)** or other PEFT methods to fine-tune the Qwen3 backbone more effectively.
* **Prompt Engineering:** Optimize the text instructions provided to the model (e.g., more detailed formatting or specific task-oriented prompting).
* **Data Augmentation:** Use image or text data augmentation techniques to enhance the diversity of the training data.
* **Hyperparameter Tuning:** Adjust learning rates, scheduler types, and batch sizes to stabilize convergence and improve results.

**Requirement:** Select **at least 3 categories** from the list above. For each chosen category, explain your intuition behind the changes, document your experiments, and provide a comparative analysis of how these modifications impacted your final accuracy on the **validation set**.

In [13]:
#SAME AS load_image_vqa_data but with option for more detailed prompt style
# - BASELINE PROMPT STYLE: same as before, just question and options
# - DETAILED PROMPT STYLE: adds instructions to look at key objects, text, and relations before deciding, to encourage more careful reasoning
def load_image_vqa_data_task3(path, answer_to_ids, prompt_style='baseline'):
  image_paths = []
  prompts = []
  answers = []

  for line in open(path, 'r'):
    jdata = json.loads(line)
    image_paths.append(jdata['image'])
    question = jdata['question']
    options = jdata['options']
    answer = answer_to_ids.get(jdata.get('answer', ''), None)

    if prompt_style == 'detailed':
      prompt = (
          "<image>\nYou are a visual multiple-choice assistant.\n"
          "Look at key objects, text, and relations before deciding.\n"
          f"Question: {question}\n"
          "Options:\n"
          f"A. {options[0]}\n"
          f"B. {options[1]}\n"
          f"C. {options[2]}\n"
          f"D. {options[3]}\n"
          "Return exactly one letter: A, B, C, or D.\n"
          "Answer:\n"
      )
    else:
      prompt = (
          "<image>\nAnswer with A, B, C, or D only.\n"
          f"Question: {question}\n"
          "Options:\n"
          f"A. {options[0]}\n"
          f"B. {options[1]}\n"
          f"C. {options[2]}\n"
          f"D. {options[3]}\n"
          "Answer:\n"
      )

    prompts.append(prompt)
    answers.append(answer)

  return {'image_paths': image_paths, 'prompts': prompts, 'answers': answers}

In [14]:
##NEW PROJECTOR VARIANT BEYOND BASIC MLP WITH LAYER NORM AND GATING, TO ENCOURAGE BETTER TRAINING STABILITY AND GRADIENT FLOW
class GatedLayerNormProjector(nn.Module):
  def __init__(self, config: LlavaConfig):
    super().__init__()
    vision_hidden = config.vision_config.hidden_size
    text_hidden = config.text_config.hidden_size
    act_name = getattr(config, 'projector_hidden_act', 'gelu')

    self.linear_1 = nn.Linear(vision_hidden, text_hidden)
    act = ACT2FN[act_name] if act_name in ACT2FN else ACT2FN['gelu']
    if isinstance(act, type):
      act = act()
    self.act = act
    self.linear_2 = nn.Linear(text_hidden, text_hidden)

    self.gate = nn.Linear(text_hidden, text_hidden)
    self.norm = nn.LayerNorm(text_hidden)

  def forward(self, image_features):
    x = self.linear_1(image_features)
    x = self.act(x)
    x = self.linear_2(x)
    gated = torch.sigmoid(self.gate(x)) * x
    return self.norm(gated + x)


In [15]:
## HELPER FUNCTION THAT WRITES PREDICTIONS TO EXPERIMENT-SPECIFIC FILES FOR TASK 3. 
#SO EACH EXPERIMENT IS KEPT SEPARATE AND CAN BE EVALUATED INDIVIDUALLY WITHOUT OVERWRITING EACH OTHER
def write_answers_to_custom_file(input_path, predictions, answer_to_ids, output_path):
  id_to_answer = {v: k for k, v in answer_to_ids.items()}
  rows = []

  for i, line in enumerate(open(input_path, 'r')):
    jdata = json.loads(line)
    jdata['answer'] = id_to_answer[int(predictions[i])]
    rows.append(jdata)

  with open(output_path, 'w') as f:
    for row in rows:
      f.write(json.dumps(row) + '\n')

In [16]:
#MAIN TRAINING AND EVALUATION FUNCTION FOR TASK 3, WITH OPTIONS FOR DIFFERENT PROJECTOR VARIANTS AND PROMPT STYLES, AND SEPARATE OUTPUT FILES FOR EACH EXPERIMENT
#PLUS HYPERPARAMETER OPTIONS FOR MORE FLEXIBLE TRAINING

def train_evaluate_task3_experiment(
    experiment_name,
    projector_variant='mlp',
    prompt_style='baseline',
    epochs=3,
    train_batch_size=2,
    eval_batch_size=2,
    max_train_samples=3000,
    learning_rate=1e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    grad_accum_steps=4,
    freeze_vision=True,
    freeze_lm=True,
):
  random.seed(0)
  torch.manual_seed(0)
  np.random.seed(0)

  answer_to_ids = {'A': 0, 'B': 1, 'C': 2, 'D': 3}

  if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'Using GPU: {torch.cuda.get_device_name(0)}')
  else:
    device = torch.device('cpu')
    print('Using CPU')

  tokenizer = AutoTokenizer.from_pretrained(TEXI_MODEL_ID, trust_remote_code=True)
  if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

  image_processor = CLIPImageProcessor.from_pretrained(CLIP_MODEL_ID)
  vision_encoder = CLIPVisionModel.from_pretrained(CLIP_MODEL_ID)

  lm_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
  language_model = AutoModelForCausalLM.from_pretrained(
      TEXI_MODEL_ID,
      trust_remote_code=True,
      torch_dtype=lm_dtype,
  )

  llava_config = LlavaConfig(
      vision_config=vision_encoder.config.to_dict(),
      text_config=language_model.config.to_dict(),
      projector_hidden_act='gelu',
  )

  if projector_variant == 'gated_ln':
    projector = GatedLayerNormProjector(llava_config)
  else:
    projector = LlavaMultiModalProjector(llava_config)

  model = CustomLLaVAForMCQ(vision_encoder, language_model, projector).to(device)

  if freeze_vision:
    for p in model.vision_encoder.parameters():
      p.requires_grad = False
  if freeze_lm:
    for p in model.language_model.parameters():
      p.requires_grad = False

  train_data = load_image_vqa_data_task3('vqa_train_student.jsonl', answer_to_ids, prompt_style=prompt_style)
  val_data = load_image_vqa_data_task3('vqa_val_student_with_image_ids.jsonl', answer_to_ids, prompt_style=prompt_style)
  test_data = load_image_vqa_data_task3('vqa_test_student_with_image_ids.jsonl', answer_to_ids, prompt_style=prompt_style)

  max_train = min(max_train_samples, len(train_data['image_paths']))
  train_samples = [
      {
          'image_path': train_data['image_paths'][i],
          'prompt': train_data['prompts'][i],
          'answer': train_data['answers'][i],
      }
      for i in range(max_train)
      if train_data['answers'][i] is not None
  ]

  trainable_params = [p for p in model.parameters() if p.requires_grad]
  optimizer = torch.optim.AdamW(trainable_params, lr=learning_rate, weight_decay=weight_decay)

  total_steps = max(1, (len(train_samples) // train_batch_size) * epochs // grad_accum_steps)
  warmup_steps = int(total_steps * warmup_ratio)

  def lr_lambda(current_step):
    if current_step < warmup_steps:
      return float(current_step) / float(max(1, warmup_steps))
    progress = float(current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    return 0.5 * (1.0 + np.cos(np.pi * progress))

  scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

  model.train()
  for epoch in range(epochs):
    random.shuffle(train_samples)
    running_loss = 0.0

    for step, start in enumerate(tqdm(range(0, len(train_samples), train_batch_size), desc=f'{experiment_name} | Epoch {epoch+1}')):
      batch_samples = train_samples[start:start + train_batch_size]
      pixel_values, input_ids, attention_mask, labels = prepare_training_batch(
          batch_samples, tokenizer, image_processor, answer_to_ids, device
      )

      outputs = model(pixel_values=pixel_values, input_ids=input_ids, attention_mask=attention_mask, labels=labels)
      loss = outputs.loss / grad_accum_steps
      loss.backward()
      running_loss += loss.item()

      if (step + 1) % grad_accum_steps == 0:
        torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)

    print(f'{experiment_name} | Epoch {epoch+1} avg loss: {running_loss / max(1, step+1):.4f}')

  processor = {
      'tokenizer': tokenizer,
      'image_processor': image_processor,
      'id_to_answer': {v: k for k, v in answer_to_ids.items()},
      'choice_token_ids': [tokenizer.encode(' ' + x, add_special_tokens=False)[-1] for x in ['A', 'B', 'C', 'D']],
  }

  val_predictions = get_prediction(model, processor, eval_batch_size, device, val_data)
  final_test_predictions = get_prediction(model,processor,eval_batch_size,device,test_data)


  val_output_path = f'vqa_val_student_final_{experiment_name}.jsonl'
  write_answers_to_custom_file('vqa_val_student_with_image_ids.jsonl', val_predictions, answer_to_ids, val_output_path)
  write_answers_to_custom_file('vqa_test_student_with_image_ids.jsonl',final_test_predictions, answer_to_ids,'vqa_test_student_final.jsonl')

  val_acc = evaluation('vqa_val_teacher.jsonl', val_output_path)
  print(f'{experiment_name} | Validation accuracy: {val_acc*100:.2f}%')

  return {'experiment': experiment_name, 'val_acc': val_acc}

In [17]:
#EXPERIMENTS:
#BASELINE: MLP projector + baseline prompts

#EXPERIMENT 1: BASELINE + DETAILED PROMPTS
#HYPOTHESIS: Adding more detailed instructions to the prompt will encourage the model to reasonmore carefully about the image and question, leading to better performance.
#exp_promt_detailed = train_evaluate_task3_experiment(
#    experiment_name='detailed_prompts',
#    projector_variant='mlp',
#    prompt_style='detailed'
#)

In [18]:
#EXPERIMENT 2: BASELINE + GATED LAYER NORM PROJECTOR
#HYPOTHESIS: Using a more sophisticated projector with gating and layer normalization will allow for better gradient flow and more effective learning of the image-text alignment, improving performance.
#exp_proj_gated_ln = train_evaluate_task3_experiment(
#    experiment_name='gated_ln_projector',
#    projector_variant='gated_ln',
#    prompt_style='baseline'
#)

In [19]:
#EXPERIMENT 3: DETAILED PROMPTS + GATED LAYER NORM PROJECTOR
#HYPOTHESIS: Combining the benefits of detailed prompts and a more advanced projector will lead to the best performance, as the model will be encouraged to reason more carefully while also having a more effective way to integrate visual information.
#exp_combined = train_evaluate_task3_experiment(
#    experiment_name='combined_detailed_gated_ln',
#    projector_variant='gated_ln',
#    prompt_style='detailed'
#)

In [20]:
#EXPERIMENT 4: DETAILED PROMPTS + GATED LAYER NORM PROJECTOR + HYPERPARAMETER TUNING
#HYPOTHESIS: Further tuning hyperparameters such as learning rate, batch size, and number of epochs will allow us to maximize the performance gains from the improved prompt and projector, leading to even higher accuracy.
#exp_combined_tuned = train_evaluate_task3_experiment(
#    experiment_name='combined_tuned',
#    projector_variant='gated_ln',
#    prompt_style='detailed',
#    epochs=5,
#    learning_rate=5e-5,
#    train_batch_size=8,
#    eval_batch_size=16,
#    grad_accum_steps=8,
#)

In [21]:
#EXPERIMENT 5: DETAILED PROMPTS + GATED LAYER NORM PROJECTOR + HYPERPARAMETER TUNING V2
#HYPOTHESIS: Further tuning hyperparameters such as learning rate, batch size, and number of epochs will allow us to maximize the performance gains from the improved prompt and projector, leading to even higher accuracy.
#exp_combined_tuned = train_evaluate_task3_experiment(
#    experiment_name='combined_exp5',
#    projector_variant='gated_ln',
#    prompt_style='detailed',
#    learning_rate=5e-5,
#)


In [22]:
#EXPERIMENT 6: DETAILED PROMPTS + GATED LAYER NORM PROJECTOR + HYPERPARAMETER TUNING V2
#HYPOTHESIS: Further tuning hyperparameters such as learning rate, batch size, and number of epochs will allow us to maximize the performance gains from the improved prompt and projector, leading to even higher accuracy.
#exp_combined_tuned = train_evaluate_task3_experiment(
#    experiment_name='combined_exp6',
#    projector_variant='gated_ln',
#    prompt_style='detailed',
#    learning_rate=1e-4,
#)

In [23]:
#EXPERIMENT 7: DETAILED PROMPTS + GATED LAYER NORM PROJECTOR + HYPERPARAMETER TUNING V2
#HYPOTHESIS: Further tuning hyperparameters such as learning rate, batch size, and number of epochs will allow us to maximize the performance gains from the improved prompt and projector, leading to even higher accuracy.
#exp_combined_tuned = train_evaluate_task3_experiment(
#    experiment_name='combined_exp7',
#    projector_variant='gated_ln',
#    prompt_style='detailed',
#    weight_decay=0.0,
#)

In [24]:
#EXPERIMENT 8: MORE EPOCHS IN BEST CONFIG=EXPERIMENT2
train_evaluate_task3_experiment(
    experiment_name='combined_ep8',
    projector_variant='gated_ln',
    prompt_style='baseline',
    epochs=10
)

Using GPU: NVIDIA L40S


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8979.06it/s]
[transformers] CLIPVisionModel LOAD REPORT from: AlpachinoNLP/LongCLIP-ViT-B-32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.embeddings.token_embedding.weight                 | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj

combined_ep8 | Epoch 1 avg loss: 1.7493


combined_ep8 | Epoch 2: 100%|██████████| 1500/1500 [03:28<00:00,  7.18it/s]


combined_ep8 | Epoch 2 avg loss: 0.7444


...

combined_ep8 | Epoch 10: 100%|██████████| 1500/1500 [03:13<00:00,  7.76it/s]


combined_ep8 | Epoch 10 avg loss: 0.0629


Predicting: 100%|██████████| 2769/2769 [04:35<00:00, 10.05it/s]

combined_ep8 | Validation accuracy: 36.70%


{'experiment': 'combined_ep8', 'val_acc': 0.36696969696969695}

In [25]:
#EXPERIMENT 9: ++MORE EPOCHS IN BEST CONFIG=EXPERIMENT2
#train_evaluate_task3_experiment(
#    experiment_name='combined_ep8',
#    projector_variant='gated_ln',
#    prompt_style='baseline',
#    epochs=15
#)

### Task 4: Performance on Test Set

For this task, no additional implementation is required. The demonstrator will execute the evaluation script against your submitted `vqa_test_student_final.jsonl` file.

Points for this section are awarded based on the **ranking of your test score** relative to all other submissions in the cohort. The score is calculated using the following logic:

$$
Score =
\begin{cases}
20 - \text{rank} + 1 & \text{if } \text{rank} \leq 10 \\
10 - \lfloor \frac{\text{rank-10}}{10} \rfloor & \text{if } \text{rank} > 10
\end{cases}
$$

In [26]:
import json

# This is to be used by the demonstrator to check the quality of your task 1
# Please do not modify it in any way; you don't need to run it
def evaluate_image_caption_alignment(gold_path,pred_path):
  gold_data = set()
  pred_data = set()
  for line in open(gold_path, 'r'):
    jdata = json.loads(line)
    gold_data.add((jdata['caption_id'],jdata['image']))
  for line in open(pred_path, 'r'):
    jdata = json.loads(line)
    pred_data.add((jdata['caption_id'],jdata['image']))
  return len(gold_data & pred_data)/len(gold_data)
acc = evaluate_image_caption_alignment('vqa_test_teacher.jsonl','vqa_test_student_final.jsonl')
print(f"Evaluation accuracy for image mapping: {acc*100.0:.2f}%")


#evaluate the VQA accuracy between the gold and predicted data
# This is exactly the same evaluation method used above, copied here only for the convenience of the demonstrator.
def evaluation(gold_path,pred_path):
  gold_data = []
  pred_data = []
  for line in open(gold_path, 'r'):
    jdata = json.loads(line)
    gold_data.append(jdata['answer'])
  for line in open(pred_path, 'r'):
    jdata = json.loads(line)
    pred_data.append(jdata['answer'])
  return sum(1 for i in range(len(gold_data)) if gold_data[i] == pred_data[i])/len(gold_data)

test_eval_acc = evaluation('vqa_test_teacher.jsonl','vqa_test_student_final.jsonl')
print(f"Final evaluation accuracy on test set: {test_eval_acc*100.0:.2f}%")

FileNotFoundError: [Errno 2] No such file or directory: 'vqa_test_teacher.jsonl'